In [ ]:
from gdrive_fsspec import GoogleDriveFileSystem
import marimo as mo

fs = GoogleDriveFileSystem(use_listings_cache=False, skip_instance_cache=True, auth_kwargs={"use_local_webserver": False})
mo.output.clear_console()

In [ ]:
import os
from pathlib import Path
from fsspec.callbacks import TqdmCallback
from concurrent.futures import ThreadPoolExecutor

# 1. Define paths and roots
DATA_ROOT = Path("/marimo/data")
VOX_DIR = DATA_ROOT / "voxceleb1"
MUSAN_DIR = DATA_ROOT / "musan"
VOX_TEST_DIR = DATA_ROOT / "voxceleb1_test"

# Setup target folders safely
DATA_ROOT.mkdir(parents=True, exist_ok=True)

# 2. Execution Targets
datasets_to_download = [
    {"remote_path": "Data_Science_Project/data/vox1_dev_wav.zip", "extract_to": VOX_DIR, "name": "voxceleb1", "extension": ".zip"},
    {"remote_path": "Data_Science_Project/data/musan.tar.gz", "extract_to": MUSAN_DIR, "name": "musan", "extension": ".tar.gz"},
    {"remote_path": "Data_Science_Project/data/vox1_test_wav.zip", "extract_to": VOX_TEST_DIR, "name": "voxceleb1_test", "extension": ".zip"}
]

def download_and_extract_worker(task):
    """Worker thread with a thread-isolated filesystem client"""
    remote_path = task["remote_path"]
    extract_to_path = Path(task["extract_to"])
    name = task["name"]
    extension = task["extension"]
    local_archive = Path(f"/marimo/{name}{extension}")

    # Short-circuit: skip if folder exists and already contains data
    if extract_to_path.exists() and any(extract_to_path.iterdir()):
        print(f"✓ {name} already exists on disk. Skipping download.")
        return

    # CRITICAL: Create an isolated Google Drive client specifically for THIS thread
    # Replace GDriveFileSystem() with the exact way you instantiated 'fs' originally
    from gdrive_fsspec import GoogleDriveFileSystem
    thread_safe_fs = GoogleDriveFileSystem()

    # 1. Thread-Isolated Download Stream
    print(f"📥 Starting parallel download for {name}...")
    tqdm_settings = {"desc": f"Downloading {name}", "unit": "B", "unit_scale": True, "unit_divisor": 1024, "leave": False}

    thread_safe_fs.get(remote_path, str(local_archive), callback=TqdmCallback(tqdm_kwargs=tqdm_settings))

    print(f"🎉 {name} extraction complete!")

# ============================================================
# PARALLEL EXECUTION MATRIX
# ============================================================
print("🚀 Initiating thread-isolated parallel download streams...")

with ThreadPoolExecutor(max_workers=3) as executor:
    list(executor.map(download_and_extract_worker, datasets_to_download))

print("\n✅ All operations completed successfully!")

In [ ]:
def finetune_waveunet_v6_final():
    # /// script
    # requires-python = ">=3.11,<3.14"
    # dependencies = [
    #     "torch", "torchaudio", "transformers",
    #     "denoisers", "speechbrain",
    #     "scikit-learn", "tqdm", "pandas",
    #     "numpy", "soundfile", "huggingface_hub",
    #     "wandb",
    # ]
    # ///

    import os
    os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
    os.environ.setdefault("WANDB_MODE", "online")

    import sys, types, re, json, random, math, enum, gc, time, urllib.request, multiprocessing, hashlib
    import importlib, importlib.machinery, importlib.util
    from pathlib import Path
    from collections import defaultdict, OrderedDict

    # ---------------- wandb ----------------
    _WANDB_ENABLED = os.environ.get("WANDB_MODE", "online") != "disabled"
    if _WANDB_ENABLED:
        try:
            import wandb
        except ImportError:
            print("[wandb] not installed — disabling.")
            _WANDB_ENABLED = False

    # ---------------- Drive ----------------
    fs = globals().get("fs", None)
    _DRIVE_ENABLED = fs is not None and os.environ.get("DRIVE_ENABLED", "true").lower() in {"1", "true", "yes"}
    _DRIVE_REMOTE_ENH = "Data_Science_Project/checkpoints/simple_enhancer_v5"   # unchanged: reuse trained enhancer
    _DRIVE_REMOTE_FUS = "Data_Science_Project/checkpoints/sa_fusion_v6_final"   # new: learned-blend architecture
    _DRIVE_SYNC_EVERY = int(os.environ.get("DRIVE_SYNC_EVERY", "10"))

    if _DRIVE_ENABLED:
        for d in (_DRIVE_REMOTE_ENH, _DRIVE_REMOTE_FUS):
            try:
                fs.makedirs(d, exist_ok=True)
            except Exception as e:
                print(f"[drive] mkdir failed for {d}: {e}")
        print(f"[drive] ready: {_DRIVE_REMOTE_ENH}, {_DRIVE_REMOTE_FUS}")
    else:
        print("[drive] disabled (fs not in scope or DRIVE_ENABLED=false).")

    # ---------------- torchvision stub ----------------
    os.environ["TRANSFORMERS_NO_TORCHVISION"] = "1"
    os.environ["USE_TORCHVISION"] = "0"
    os.environ["TRANSFORMERS_NO_TORCHVISION_IMAGE"] = "1"

    _BROKEN_TV = True
    try:
        import torchvision  # noqa
        from torchvision.transforms import InterpolationMode  # noqa
        _BROKEN_TV = False
    except Exception:
        pass

    def _install_stub(name, is_package=True):
        mod = types.ModuleType(name)
        spec = importlib.machinery.ModuleSpec(name=name, loader=None, is_package=is_package)
        if is_package:
            spec.submodule_search_locations = []
        mod.__spec__ = spec
        mod.__loader__ = None
        mod.__package__ = name if is_package else name.rpartition(".")[0]
        mod.__path__ = [] if is_package else None
        mod.__file__ = f"<stub:{name}>"
        mod.__version__ = "0.0.0-stub"
        sys.modules[name] = mod
        return mod

    if _BROKEN_TV:
        fake = _install_stub("torchvision", True)
        tr = _install_stub("torchvision.transforms", True)

        class InterpolationMode(enum.Enum):
            NEAREST = "nearest"; NEAREST_EXACT = "nearest-exact"; BILINEAR = "bilinear"
            BICUBIC = "bicubic"; BOX = "box"; HAMMING = "hamming"; LANCZOS = "lanczos"

        tr.InterpolationMode = InterpolationMode
        for c in ("Compose", "Resize", "CenterCrop", "RandomCrop", "Normalize", "ToTensor",
                  "PILToTensor", "ConvertImageDtype", "RandomResizedCrop", "RandomHorizontalFlip",
                  "ColorJitter", "Grayscale", "RandomApply", "Lambda", "ToPILImage"):
            setattr(tr, c, type(c, (), {}))
        tr_func = _install_stub("torchvision.transforms.functional", False)
        for fn in ("resize", "center_crop", "normalize", "to_tensor", "pil_to_tensor", "to_pil_image",
                   "rgb_to_grayscale", "crop", "pad", "hflip", "vflip", "rotate", "adjust_brightness",
                   "adjust_contrast", "adjust_saturation", "adjust_hue", "convert_image_dtype", "gaussian_blur"):
            setattr(tr_func, fn, lambda *a, **k: None)
        tr.functional = tr_func
        tr_v2 = _install_stub("torchvision.transforms.v2", True)
        tr_v2.InterpolationMode = InterpolationMode
        for c in ("Compose", "Transform", "ToDtype", "ToImage", "Normalize", "Resize", "CenterCrop", "RandomCrop"):
            setattr(tr_v2, c, type(c, (), {}))
        tr_v2_func = _install_stub("torchvision.transforms.v2.functional", False)
        tr_v2.functional = tr_v2_func; tr.v2 = tr_v2; fake.transforms = tr
        io_mod = _install_stub("torchvision.io", False)
        io_mod.ImageReadMode = type("ImageReadMode", (), {"UNCHANGED": 0, "GRAY": 1, "GRAY_ALPHA": 2, "RGB": 3, "RGBA": 4})
        for fn in ("decode_image", "read_image", "write_image", "encode_jpeg", "encode_png", "decode_jpeg", "decode_png"):
            setattr(io_mod, fn, lambda *a, **k: None)
        fake.io = io_mod
        fake.models = _install_stub("torchvision.models", True)
        ops = _install_stub("torchvision.ops", False)
        for fn in ("nms", "batched_nms", "nms_rotated", "box_convert", "box_iou", "generalized_box_iou",
                   "roi_align", "roi_pool", "ps_roi_align", "ps_roi_pool", "sigmoid_focal_loss", "deform_conv2d"):
            setattr(ops, fn, lambda *a, **k: None)
        for c in ("RoIAlign", "RoIPool", "PSRoIAlign", "PSRoIPool", "DeformConv2d"):
            setattr(ops, c, type(c, (), {}))
        fake.ops = ops
        tv = _install_stub("torchvision.tv_tensors", False)
        for c in ("TVTensor", "Image", "BoundingBoxes", "Mask", "Video", "BoundingBoxFormat"):
            setattr(tv, c, type(c, (), {}))
        fake.tv_tensors = tv
        utils = _install_stub("torchvision.utils", False)
        for fn in ("make_grid", "save_image", "draw_bounding_boxes", "draw_segmentation_masks"):
            setattr(utils, fn, lambda *a, **k: None)
        fake.utils = utils
        assert importlib.util.find_spec("torchvision") is not None
        print("[stub] torchvision ok")

    _MP_CTX = None
    if os.name == "posix":
        try:
            _MP_CTX = multiprocessing.get_context("fork"); print("Using fork multiprocessing context")
        except Exception:
            _MP_CTX = None

    import torch.nn as nn
    _orig_getattr = nn.Module.__getattr__
    nn.Module.__getattr__ = lambda self, name: {} if name == "all_tied_weights_keys" else _orig_getattr(self, name)

    import numpy as np, pandas as pd, torch, torch.nn.functional as F, torchaudio
    from torch.utils.data import Dataset, Sampler, DataLoader
    from tqdm import tqdm
    from sklearn.metrics import roc_curve
    from huggingface_hub import hf_hub_download
    from denoisers import WaveUNetModel, WaveUNetConfig

    torch.backends.cudnn.benchmark = False

    def make_autocast(en):
        try: return torch.autocast(device_type="cuda", dtype=torch.float16, enabled=en)
        except (TypeError, AttributeError): return torch.cuda.amp.autocast(enabled=en)
    def make_scaler(en):
        try: return torch.amp.GradScaler("cuda", enabled=en)
        except (TypeError, AttributeError): return torch.cuda.amp.GradScaler(enabled=en)
    def sseed(*p):
        t = "||".join(str(x) for x in p)
        return int.from_bytes(hashlib.sha256(t.encode()).digest()[:8], "little")

    # =========================================================================
    # CONFIG
    # =========================================================================
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    USE_CUDA = DEVICE.startswith("cuda")
    SB_DEVICE = f"cuda:{torch.cuda.current_device()}" if USE_CUDA else "cpu"
    print(f"Device: {DEVICE}")
    if USE_CUDA:
        props = torch.cuda.get_device_properties(0)
        print(f"GPU: {props.name} | VRAM: {props.total_memory/1e9:.1f} GB")
        CHUNK_BS = 32 if props.total_memory / 1e9 >= 40 else (16 if props.total_memory / 1e9 >= 20 else 8)
    else:
        CHUNK_BS = 4

    DEV_ROOT = "/marimo/voxceleb1_dev"
    TEST_ROOT = "/marimo/voxceleb1_test/wav"
    TRIAL_FILE = "/marimo/veri_test.txt"
    MUSAN_ROOT = "/marimo/musan"
    ECAPA_DIR = "/marimo/ecapa_voxceleb"

    ENH_CKPT_DIR = "/marimo/simple_enhancer_checkpoints_v5"       # unchanged: reuse trained enhancer
    FUS_CKPT_DIR = "/marimo/sa_fusion_checkpoints_v6_final"       # new: learned-blend architecture
    os.makedirs(ENH_CKPT_DIR, exist_ok=True)
    os.makedirs(FUS_CKPT_DIR, exist_ok=True)

    SR = 16000; MIN_SEC = 4.0; MAX_SEC = 10.0
    SNR_RANGE = (-20.0, 20.0)
    CLEAN_FRAC = 0.15; CLEAN_SNR = 30.0
    P_SPEAKERS = 12; K_UTTS = 4

    # ---- 5-loss set (see header comment for what was cut) ----
    LAMBDA_STFT = 0.40
    LAMBDA_SI = 0.05
    LAMBDA_ID = 0.40
    LAMBDA_PROTO = 0.30
    LAMBDA_CLEAN = 0.15
    PROTO_TEMPERATURE = 0.10

    ENH_EPOCHS = 30
    ENH_STEPS_PER_EP = 300
    ENH_LR = 8e-5
    ENH_GRAD_CLIP = 5.0
    ENH_WARMUP_STEPS = 150

    FUS_EPOCHS = 20
    FUS_STEPS_PER_EP = 300
    FUS_LR = 5e-4
    FUS_GRAD_CLIP = 5.0
    FUS_WARMUP_STEPS = 150
    TRIPLET_MARGIN = 0.25
    LAMBDA_FUS_PRESERVE = 0.10
    FUS_PRESERVE_SNR_THRESHOLD = 15.0
    FUS_PRESERVE_SNR_TEMP = 3.0

    SA_NUM_HEADS = 4; SA_NUM_LAYERS = 2; SA_DROPOUT = 0.1

    N_VAL_SPEAKERS = 40; N_VAL_UTTS_PER_SPK = 4
    EVAL_BATCH = 8
    VAL_SNR_WEIGHTS = [(-20.0, 0.15), (-10.0, 0.15), (-5.0, 0.15), (0.0, 0.25), (5.0, 0.15), (15.0, 0.15)]
    EVAL_EVERY_N_EPOCHS = 2

    NOISE_SUBDIRS = [("babble", "speech"), ("music", "music"), ("noise", "noise")]
    MUSAN_TEST_RATIO = 0.30; MUSAN_SPLIT_SEED_BASE = 1234
    EVAL_SNRS = [-20, -15, -10, -5, 0, 5, 10, 15, 20]
    PAPER_SNRS = [0, -5, -10, -15, -20]
    FUS_VAL_SNRS = [-20.0, -15.0, -10.0, -5.0, 0.0, 5.0, 10.0]

    # =========================================================================
    # SHARED UTILS
    # =========================================================================
    AUDIO_EXTS = {".wav", ".flac", ".ogg", ".mp3"}
    _AC = OrderedDict()
    def _cp(k, v):
        if k in _AC: _AC.move_to_end(k); return
        _AC[k] = v
        while len(_AC) > 5000: _AC.popitem(last=False)
    def list_audio(root):
        if not os.path.isdir(root): return []
        return sorted(str(p) for p in Path(root).rglob("*") if p.is_file() and p.suffix.lower() in AUDIO_EXTS)
    def extract_spk(p):
        for part in Path(p).parts:
            if part.startswith("id") and len(part) >= 5: return part
        return Path(p).parent.name
    def load_wav(p, sr_t=SR):
        k = (str(p), sr_t)
        v = _AC.get(k)
        if v is not None: _AC.move_to_end(k); return v
        try: wav, sr = torchaudio.load(str(p), backend="soundfile")
        except Exception: wav, sr = torchaudio.load(str(p))
        if sr != sr_t: wav = torchaudio.functional.resample(wav, sr, sr_t)
        if wav.dim() == 2: wav = wav.mean(dim=0)
        wav = wav.float().contiguous(); _cp(k, wav); return wav
    def rms(x, eps=1e-8): return torch.sqrt(torch.mean(x ** 2) + eps)
    def mix_at_snr(c, n, snr):
        if n.numel() < c.numel(): n = n.repeat(int(np.ceil(c.numel() / n.numel())))
        if n.numel() > c.numel():
            s = random.randint(0, n.numel() - c.numel()); n = n[s:s + c.numel()]
        sc = rms(c) / (rms(n) * (10 ** (snr / 20)) + 1e-8)
        nc = sc * n; no = c + nc
        pk = torch.maximum(c.abs().max(), no.abs().max()).clamp_min(1e-6)
        return no / pk, c / pk, nc / pk
    def mix_det(c, n, snr, seed_key):
        r = random.Random(sseed(*seed_key))
        if n.numel() < c.numel(): n = n.repeat(int(np.ceil(c.numel() / n.numel())))
        if n.numel() > c.numel():
            s = r.randint(0, n.numel() - c.numel()); n = n[s:s + c.numel()]
        else:
            n = n[:c.numel()]
        sc = rms(c) / (rms(n) * (10 ** (snr / 20)) + 1e-8)
        nc = sc * n; no = c + nc
        pk = torch.maximum(c.abs().max(), no.abs().max()).clamp_min(1e-6)
        return no / pk, c / pk, nc / pk
    def compute_eer(sc, lb):
        s = np.asarray(sc, dtype=np.float64); l = np.asarray(lb, dtype=np.int64)
        if len(np.unique(l)) < 2: return 50.0
        fpr, tpr, _ = roc_curve(l, s); fnr = 1 - tpr
        i = np.nanargmin(np.abs(fnr - fpr)); return float((fpr[i] + fnr[i]) / 2 * 100)
    def split_strat(files, r=0.30, seed=42):
        files = sorted(files); rng = random.Random(seed)
        sh = files.copy(); rng.shuffle(sh)
        n = max(1, int(len(sh) * r)); return sh[n:], sh[:n]
    def _unwrap(o):
        if hasattr(o, "audio"): return o.audio
        if hasattr(o, "waveform"): return o.waveform
        if isinstance(o, (list, tuple)): return o[0]
        if isinstance(o, dict): return next(v for v in o.values() if torch.is_tensor(v))
        return o

    # ---- MUSAN split ----
    print("\nMUSAN split:")
    noise_train_banks = {}; noise_test_banks = {}
    for i, (nt, sub) in enumerate(NOISE_SUBDIRS):
        files = list_audio(os.path.join(MUSAN_ROOT, sub))
        tr, te = split_strat(files, MUSAN_TEST_RATIO, MUSAN_SPLIT_SEED_BASE + i)
        assert not (set(tr) & set(te))
        noise_train_banks[nt] = tr; noise_test_banks[nt] = te
        print(f"  {nt:8s}: train={len(tr):5d}  test={len(te):5d}")
    train_noise = []
    for nt in noise_train_banks: train_noise.extend(noise_train_banks[nt])
    random.seed(43); random.shuffle(train_noise)

    # ---- ECAPA (frozen throughout) ----
    print("\nLoading ECAPA ...")
    from speechbrain.inference.speaker import EncoderClassifier
    ecapa = EncoderClassifier.from_hparams(source="speechbrain/spkrec-ecapa-voxceleb",
                                            savedir=ECAPA_DIR, run_opts={"device": SB_DEVICE})
    for p in ecapa.parameters(): p.requires_grad_(False)
    ecapa.eval()
    @torch.no_grad()
    def ecapa_embed(w, wl=None):
        return F.normalize(ecapa.encode_batch(w, wl).squeeze(1).float(), dim=-1)
    def ecapa_embed_grad(w, wl=None):
        return F.normalize(ecapa.encode_batch(w, wl).squeeze(1).float(), dim=-1)
    with torch.no_grad():
        EMB_DIM = ecapa_embed(torch.zeros(1, SR, device=DEVICE), torch.ones(1, device=DEVICE)).shape[-1]
    print(f"ECAPA dim: {EMB_DIM}")

    # ---- Wave-U-Net ----
    MODEL_ID = "wrice/waveunet-vctk-48khz-audiomentations"
    print("Loading Wave-U-Net ...")
    with open(hf_hub_download(MODEL_ID, "config.json")) as f: cd = json.load(f)
    cfg = WaveUNetConfig(**cd); MODEL_SR = cd["sample_rate"]; MAX_LEN = cd["max_length"]
    enhancer = WaveUNetModel(cfg).to(DEVICE)
    sd = torch.load(hf_hub_download(MODEL_ID, "pytorch_model.bin"), map_location="cpu", weights_only=False)
    for key in ("state_dict", "model", "model_state_dict"):
        if isinstance(sd, dict) and key in sd and isinstance(sd[key], dict): sd = sd[key]; break
    new_sd = {}
    for k, v in sd.items():
        k2 = k
        if (".encoder_layers." in k or ".decoder_layers." in k) and ".batch_norm." in k:
            k2 = k2.replace(".batch_norm.", ".norm.norm.")
        m = re.match(r"^(.*\.middle\.\d+)\.(weight|bias|running_mean|running_var|num_batches_tracked)$", k2)
        if m: k2 = f"{m.group(1)}.norm.{m.group(2)}"
        new_sd[k2] = v
    missing, unexpected = enhancer.load_state_dict(new_sd, strict=False)
    print(f"  WUN: missing={len(missing)} unexpected={len(unexpected)}")
    for p in enhancer.parameters(): p.requires_grad = True

    # =========================================================================
    # SHARED: chunked enhancement (no gate, direct output)
    # =========================================================================
    def _enhance_chunked(wav, wl, model=None, amp=True, cbs=None):
        if model is None: model = enhancer
        if cbs is None: cbs = CHUNK_BS
        B, Tm = wav.shape
        r16 = (wl.detach().cpu().float() * Tm).round().long()
        r48 = (r16.float() * MODEL_SR / SR).round().long()
        wav48 = torchaudio.functional.resample(wav, SR, MODEL_SR)
        T48m = wav48.shape[-1]

        chunks = []; owners = []
        for b in range(B):
            Lb = int(r48[b].item())
            if Lb <= 0: continue
            w = wav48[b, :Lb]
            pad = (MAX_LEN - w.numel() % MAX_LEN) % MAX_LEN
            wp = F.pad(w, (0, pad)) if pad else w
            n = wp.numel() // MAX_LEN
            for i in range(n):
                chunks.append(wp[i * MAX_LEN:(i + 1) * MAX_LEN]); owners.append((b, i))

        if not chunks:
            return wav

        chunks = torch.stack(chunks).unsqueeze(1)
        enh_chunks = []
        with make_autocast(amp and USE_CUDA):
            for i in range(0, chunks.shape[0], cbs):
                c = chunks[i:i + cbs]
                out = model(c)
                noise = _unwrap(out).reshape(c.shape[0], -1)[:, :MAX_LEN].float()
                enh_chunks.append(c.squeeze(1).float() - noise)
        enh_chunks = torch.cat(enh_chunks, dim=0)

        per = [[] for _ in range(B)]
        for idx, (b, _) in enumerate(owners): per[b].append(enh_chunks[idx])

        out_list = []
        for b in range(B):
            e = torch.cat(per[b], dim=0)[:int(r48[b].item())] if per[b] else torch.zeros(0, device=wav.device)
            if e.numel() < T48m: e = F.pad(e, (0, T48m - e.numel()))
            else: e = e[:T48m]
            out_list.append(e)
        e48 = torch.stack(out_list, dim=0)
        e16 = torchaudio.functional.resample(e48, MODEL_SR, SR)
        if e16.shape[-1] > Tm: e16 = e16[..., :Tm]
        elif e16.shape[-1] < Tm: e16 = F.pad(e16, (0, Tm - e16.shape[-1]))
        return e16

    # =========================================================================
    # LOSSES
    # =========================================================================
    def masked_si_sdr(est, tgt, wl, eps=1e-8):
        B, T = est.shape
        rl = (wl * T).round().long().clamp(min=2)
        ar = torch.arange(T, device=est.device).unsqueeze(0)
        mask = (ar < rl.unsqueeze(1)).float()
        n = mask.sum(-1, keepdim=True).clamp_min(1.0)
        em = est - (est * mask).sum(-1, keepdim=True) / n
        tm = tgt - (tgt * mask).sum(-1, keepdim=True) / n
        em = em * mask; tm = tm * mask
        a = (em * tm).sum(-1, keepdim=True) / (tm.pow(2).sum(-1, keepdim=True) + eps)
        proj = a * tm; noise = em - proj
        ratio = proj.pow(2).sum(-1) / (noise.pow(2).sum(-1) + eps)
        valid = (rl >= 100).float()
        if valid.sum() < 1: return torch.tensor(0.0, device=est.device, requires_grad=True)
        sdr = -10.0 * torch.log10(ratio + eps)
        return (sdr * valid).sum() / valid.sum().clamp_min(1.0)

    def mrstft(est, tgt, ffts=(512, 1024, 2048), hops=(50, 120, 240), wins=(240, 600, 1200)):
        T = est.shape[-1]; total = 0.0; n_res = 0
        for nf, hp, wn in zip(ffts, hops, wins):
            if T < wn: continue
            window = torch.hann_window(wn, device=est.device)
            es = torch.stft(est, n_fft=nf, hop_length=hp, win_length=wn, window=window, return_complex=True)
            ts = torch.stft(tgt, n_fft=nf, hop_length=hp, win_length=wn, window=window, return_complex=True)
            em = torch.clamp(es.abs(), min=1e-7); tm = torch.clamp(ts.abs(), min=1e-7)
            sc = torch.norm(tm - em, p="fro", dim=(-2, -1)) / torch.norm(tm, p="fro", dim=(-2, -1)).clamp_min(1e-6)
            ml = F.l1_loss(torch.log(em), torch.log(tm))
            total = total + sc.mean() + ml; n_res += 1
        if n_res == 0: return torch.zeros((), device=est.device)
        return total / n_res

    def clean_preserve(out, noisy, snrs, wl, th=9.0):
        w = torch.sigmoid((snrs - th) / 5.0)
        B, T = out.shape
        rl = (wl * T).round().long().clamp(min=2)
        ar = torch.arange(T, device=out.device).unsqueeze(0)
        mask = (ar < rl.unsqueeze(1)).float()
        n = mask.sum(-1).clamp_min(1.0)
        diff = (out - noisy).abs() * mask
        return ((diff.sum(-1) / n) * w).sum() / w.sum().clamp_min(1e-6)

    def identity_loss(a, b):
        return (1.0 - (a * b).sum(-1)).mean()

    def prototypical_loss(emb, p, k, temp=PROTO_TEMPERATURE):
        N, D = emb.shape
        if N != p * k: return torch.zeros((), device=emb.device)
        e = emb.view(p, k, D)
        sum_c = e.sum(1)
        loo = (sum_c.unsqueeze(1) - e) / max(1, (k - 1))
        loo = F.normalize(loo, dim=-1); fc = F.normalize(sum_c, dim=-1)
        ef = e.reshape(p * k, D)
        sims = (ef @ fc.t()) / temp
        loo_f = loo.reshape(p * k, D)
        self_sim = (ef * loo_f).sum(-1) / temp
        own = torch.arange(p, device=emb.device).repeat_interleave(k)
        sims[torch.arange(p * k, device=emb.device), own] = self_sim
        return F.cross_entropy(sims, own)

    # =========================================================================
    # DATASET (shared)
    # =========================================================================
    class VoxDataset(Dataset):
        def __init__(self, files, spk_to_idx, noise_files, snr_range,
                     balance_noise_types=False, noise_banks=None):
            self.files = files
            self.labels = [spk_to_idx[extract_spk(f)] for f in files]
            self.noise_files = noise_files; self.snr_range = snr_range
            # balance_noise_types=True: pick the noise TYPE uniformly first,
            # then a file within that type -- instead of sampling from the
            # pooled file list, which under-weights babble (fewer MUSAN files
            # than music/noise). Default False keeps the original (enhancer)
            # behavior exactly as it was.
            self.balance_noise_types = balance_noise_types
            self.noise_banks = noise_banks  # {"babble": [...], "music": [...], "noise": [...]}
            if self.balance_noise_types:
                assert self.noise_banks, "noise_banks required when balance_noise_types=True"
                self.noise_types = sorted(self.noise_banks.keys())
        def __len__(self): return len(self.files)
        def _pick_noise_file(self):
            if self.balance_noise_types:
                nt = random.choice(self.noise_types)
                return random.choice(self.noise_banks[nt])
            return random.choice(self.noise_files)
        def __getitem__(self, idx):
            seg = int(random.uniform(MIN_SEC, MAX_SEC) * SR)
            cf = load_wav(self.files[idx], SR); n = cf.numel()
            clean = cf[random.randint(0, n - seg):][:seg].clone() if n >= seg else F.pad(cf, (0, seg - n))
            if random.random() < CLEAN_FRAC:
                return clean.clone(), clean.clone(), torch.zeros_like(clean), self.labels[idx], CLEAN_SNR
            nf = load_wav(self._pick_noise_file(), SR); n = nf.numel()
            noise = nf[random.randint(0, n - seg):][:seg].clone() if n >= seg else F.pad(nf, (0, seg - n))
            snr = random.uniform(*self.snr_range)
            noisy, clean, nc = mix_at_snr(clean, noise, snr)
            return noisy, clean, nc, self.labels[idx], snr

    def collate_v(batch):
        nl, cl, ncl, lb, sn = zip(*batch)
        L = torch.tensor([w.numel() for w in nl], dtype=torch.long)
        m = int(L.max().item())
        return (torch.stack([F.pad(w, (0, m - w.numel())) for w in nl]),
                torch.stack([F.pad(w, (0, m - w.numel())) for w in cl]),
                torch.stack([F.pad(w, (0, m - w.numel())) for w in ncl]),
                torch.tensor(lb, dtype=torch.long), L.float() / m,
                torch.tensor(sn, dtype=torch.float32))

    def collate_v_simple(batch):
        nl, cl, _, lb, sn = zip(*batch)
        L = torch.tensor([w.numel() for w in nl], dtype=torch.long)
        m = int(L.max().item())
        return (torch.stack([F.pad(w, (0, m - w.numel())) for w in nl]),
                torch.stack([F.pad(w, (0, m - w.numel())) for w in cl]),
                torch.tensor(lb, dtype=torch.long), L.float() / m,
                torch.tensor(sn, dtype=torch.float32))

    class PKBatchSampler(Sampler):
        def __init__(self, ds, p, k, steps):
            self.p = p; self.k = k; self.steps = steps
            by = defaultdict(list)
            for i, lab in enumerate(ds.labels): by[lab].append(i)
            self.by = {lab: i for lab, i in by.items() if len(i) >= k}
            self.spks = sorted(self.by.keys())
        def __iter__(self):
            for _ in range(self.steps):
                spks = random.sample(self.spks, self.p); batch = []
                for s in spks: batch.extend(random.sample(self.by[s], self.k))
                yield batch
        def __len__(self): return self.steps

    # ---- Speaker split ----
    print("\nScanning VoxCeleb1 dev ...")
    by_spk = defaultdict(list)
    for f in list_audio(DEV_ROOT): by_spk[extract_spk(f)].append(f)
    all_spks = sorted([s for s, v in by_spk.items() if len(v) >= K_UTTS])
    random.seed(42); random.shuffle(all_spks)
    val_spks = set(all_spks[:N_VAL_SPEAKERS]); train_spks = set(all_spks[N_VAL_SPEAKERS:])
    train_files = [f for s in train_spks for f in by_spk[s]]
    spk_to_idx = {s: i for i, s in enumerate(sorted(train_spks))}
    print(f"Train: {len(train_spks)} spk | Val: {len(val_spks)} spk | Utter: {len(train_files)}")

    def build_val_trials():
        trials = []; rng = random.Random(123); vl = sorted(val_spks)
        for spk in vl:
            utts = sorted(by_spk[spk])
            if len(utts) < 2: continue
            enr = utts[0]
            for tst in utts[1:min(1 + N_VAL_UTTS_PER_SPK, len(utts))]: trials.append((enr, tst, 1))
            others = [s for s in vl if s != spk]
            for _ in range(min(N_VAL_UTTS_PER_SPK, len(others))):
                imp = rng.choice(others); trials.append((enr, rng.choice(by_spk[imp]), 0))
        return trials
    val_trials = build_val_trials()
    VAL_ENR = sorted({e for e, _, _ in val_trials}); VAL_TST = sorted({t for _, t, _ in val_trials})
    print(f"Val trials: {len(val_trials)}")

    # =========================================================================
    # DRIVE RESUME HELPER (mirrors v4_long logic, reused for both stages)
    # =========================================================================
    def drive_pull_latest(local_resume, remote_dir):
        """If Drive holds a newer resume_epochN.pt than the local copy, download it."""
        if not _DRIVE_ENABLED:
            return
        remote_latest_epoch = -1; remote_latest_name = None
        try:
            entries = fs.ls(remote_dir, detail=False)
            for e in entries:
                name = e.split("/")[-1]
                m = re.match(r"^resume_epoch(\d+)\.pt$", name)
                if m:
                    ep = int(m.group(1))
                    if ep > remote_latest_epoch:
                        remote_latest_epoch = ep; remote_latest_name = name
            if remote_latest_name:
                print(f"[drive] {remote_dir}: found {remote_latest_name} (epoch {remote_latest_epoch})")
        except Exception as e:
            print(f"[drive] {remote_dir}: ls failed: {e}")
            return

        local_epoch = -1
        if os.path.isfile(local_resume):
            try:
                ck_probe = torch.load(local_resume, map_location="cpu", weights_only=False)
                local_epoch = int(ck_probe.get("epoch", -1)); del ck_probe
            except Exception as e:
                print(f"[resume] could not read local resume: {e}")

        if remote_latest_name is not None and remote_latest_epoch > local_epoch:
            print(f"[resume] Drive is newer, downloading {remote_latest_name} ...")
            try:
                fs.get(f"{remote_dir}/{remote_latest_name}", local_resume)
                print("[resume] downloaded.")
            except Exception as e:
                print(f"[resume] download failed: {e}")

    def drive_sync(local_resume, remote_dir, epoch, best_path=None, best_remote_name=None):
        if not _DRIVE_ENABLED:
            return
        remote_name = f"resume_epoch{epoch + 1}.pt"
        try:
            fs.put(local_resume, f"{remote_dir}/{remote_name}")
            print(f"  [drive] uploaded {remote_name}")
        except Exception as e:
            print(f"  [drive] resume upload failed: {e}")
        if best_path is not None and os.path.isfile(best_path):
            try:
                fs.put(best_path, f"{remote_dir}/{best_remote_name}")
            except Exception as e:
                print(f"  [drive] best upload failed: {e}")

    # =========================================================================
    # =========== PART 1: TRAIN SIMPLIFIED ENHANCER ===========
    # =========================================================================
    def train_enhancer():
        print("\n" + "=" * 78 + "\nPART 1: SIMPLIFIED ENHANCER (5 losses, no gate)\n" + "=" * 78)
        print(f"  Losses: STFT={LAMBDA_STFT}  SI={LAMBDA_SI}  ID={LAMBDA_ID}  "
              f"PROTO={LAMBDA_PROTO}  CLEAN={LAMBDA_CLEAN}")

        train_ds = VoxDataset(train_files, spk_to_idx, train_noise, SNR_RANGE)
        sampler = PKBatchSampler(train_ds, P_SPEAKERS, K_UTTS, ENH_STEPS_PER_EP)
        nw = 6 if _MP_CTX is not None else 0
        lk = dict(batch_sampler=sampler, num_workers=nw, pin_memory=True,
                  persistent_workers=(nw > 0), collate_fn=collate_v)
        if _MP_CTX is not None: lk["multiprocessing_context"] = _MP_CTX
        loader = DataLoader(train_ds, **lk)

        opt = torch.optim.AdamW(enhancer.parameters(), lr=ENH_LR, weight_decay=1e-5)
        total_steps = ENH_EPOCHS * ENH_STEPS_PER_EP
        def lr_lam(s):
            if s < ENH_WARMUP_STEPS: return s / max(1, ENH_WARMUP_STEPS)
            p = (s - ENH_WARMUP_STEPS) / max(1, total_steps - ENH_WARMUP_STEPS)
            return 0.5 * (1.0 + math.cos(math.pi * p))
        sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lam)
        scaler = make_scaler(USE_CUDA)

        local_resume = os.path.join(ENH_CKPT_DIR, "resume_latest.pt")
        best_path = os.path.join(ENH_CKPT_DIR, "simple_enhancer_best.pt")
        start_epoch = 0; best_val = float("inf")

        drive_pull_latest(local_resume, _DRIVE_REMOTE_ENH)
        if os.path.isfile(local_resume):
            try:
                ck = torch.load(local_resume, map_location="cpu", weights_only=False)
                enhancer.load_state_dict(ck["enhancer_state_dict"])
                opt.load_state_dict(ck["optimizer_state_dict"])
                sched.load_state_dict(ck["scheduler_state_dict"])
                if ck.get("scaler_state_dict"):
                    try: scaler.load_state_dict(ck["scaler_state_dict"])
                    except Exception: pass
                start_epoch = int(ck.get("epoch", -1)) + 1
                best_val = float(ck.get("best_val", float("inf")))
                print(f"[resume] enhancer: epoch {start_epoch}, best_val={best_val:.2f}")
            except Exception as e:
                print(f"[resume] enhancer: failed ({e}), starting fresh")

        @torch.inference_mode()
        def evaluate_val(snr):
            enhancer.eval()
            enr_cache = {}
            for p in VAL_ENR:
                w = load_wav(p, SR).unsqueeze(0).to(DEVICE)
                wl = torch.ones(1, device=DEVICE)
                enr_cache[p] = ecapa_embed(w, wl).cpu()[0]
            tst_cache = {}
            for i in range(0, len(VAL_TST), EVAL_BATCH):
                batch = VAL_TST[i:i + EVAL_BATCH]
                wavs = [load_wav(p, SR) for p in batch]
                if snr is not None:
                    mixed = []
                    for w in wavs:
                        nf = load_wav(random.choice(train_noise), SR)
                        if nf.numel() < w.numel(): nf = nf.repeat(int(np.ceil(w.numel() / nf.numel())))
                        if nf.numel() > w.numel():
                            s = random.randint(0, nf.numel() - w.numel()); nf = nf[s:s + w.numel()]
                        no, _, _ = mix_at_snr(w, nf, snr)
                        mixed.append(no)
                    wavs = mixed
                L = torch.tensor([w.numel() for w in wavs], dtype=torch.long)
                ml = int(L.max().item()); wl = (L.float() / ml).to(DEVICE)
                pad = torch.stack([F.pad(w, (0, ml - w.numel())) for w in wavs]).to(DEVICE)
                enh = _enhance_chunked(pad, wl, amp=False).float()
                emb = ecapa_embed(enh, wl).cpu()
                for j, p in enumerate(batch): tst_cache[p] = emb[j]
            enhancer.train()
            sc = []; lb = []
            for e, t, l in val_trials:
                ee = enr_cache[e].to(DEVICE); te = tst_cache[t].to(DEVICE)
                sc.append(F.cosine_similarity(ee.unsqueeze(0), te.unsqueeze(0)).item())
                lb.append(l)
            return compute_eer(sc, lb)

        t0 = time.time()
        best_state = None
        for epoch in range(start_epoch, ENH_EPOCHS):
            enhancer.train()
            m = defaultdict(float)
            pbar = tqdm(loader, desc=f"Enh {epoch + 1}/{ENH_EPOCHS}")
            opt.zero_grad(set_to_none=True)

            for step_i, (noisy, clean, noise_gt, labels, wl, snrs) in enumerate(pbar):
                noisy = noisy.to(DEVICE, non_blocking=True)
                clean = clean.to(DEVICE, non_blocking=True)
                labels = labels.to(DEVICE, non_blocking=True)
                wl = wl.to(DEVICE, non_blocking=True)
                snrs = snrs.to(DEVICE, non_blocking=True)

                with make_autocast(USE_CUDA):
                    enhanced = _enhance_chunked(noisy, wl, amp=True).float()

                loss_stft = mrstft(enhanced, clean)
                loss_si = masked_si_sdr(enhanced, clean, wl)
                loss_clean = clean_preserve(enhanced, noisy, snrs, wl)

                with torch.no_grad():
                    emb_clean = ecapa_embed(clean, wl)
                emb_enh = ecapa_embed_grad(enhanced, wl)
                loss_id = identity_loss(emb_clean, emb_enh)
                loss_proto = prototypical_loss(emb_enh, P_SPEAKERS, K_UTTS)

                total = (LAMBDA_STFT * loss_stft + LAMBDA_SI * loss_si
                         + LAMBDA_ID * loss_id + LAMBDA_PROTO * loss_proto
                         + LAMBDA_CLEAN * loss_clean)

                if not torch.isfinite(total):
                    opt.zero_grad(set_to_none=True); continue

                scaler.scale(total).backward()
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(enhancer.parameters(), ENH_GRAD_CLIP)
                prev = scaler.get_scale()
                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
                if scaler.get_scale() >= prev: sched.step()

                for k, v in [("total", total.item()), ("stft", loss_stft.item()), ("si", loss_si.item()),
                             ("id", loss_id.item()), ("proto", loss_proto.item()), ("clean", loss_clean.item())]:
                    m[k] += v
                with torch.no_grad():
                    m["gen_cos"] += (emb_enh * emb_clean).sum(-1).mean().item()

                if _WANDB_ENABLED:
                    global_step = epoch * ENH_STEPS_PER_EP + step_i
                    wandb.log({
                        "enh/step": global_step,
                        "enh/loss_total": total.item(),
                        "enh/loss_stft": loss_stft.item(),
                        "enh/loss_si": loss_si.item(),
                        "enh/loss_id": loss_id.item(),
                        "enh/loss_proto": loss_proto.item(),
                        "enh/loss_clean": loss_clean.item(),
                        "enh/lr": opt.param_groups[0]["lr"],
                    }, commit=True)

                pbar.set_postfix({"L": f"{total.item():.3f}", "stft": f"{loss_stft.item():.3f}",
                                  "id": f"{loss_id.item():.3f}", "prot": f"{loss_proto.item():.3f}",
                                  "clean": f"{loss_clean.item():.3f}"})

            n = max(1, len(pbar))
            print(f"\nEnh Epoch {epoch + 1}: " + " | ".join(f"{k}={v / n:.4f}" for k, v in m.items()))

            if _WANDB_ENABLED:
                wandb.log({
                    "enh/epoch": epoch + 1,
                    "enh/epoch_loss_total": m["total"] / n,
                    "enh/epoch_loss_stft": m["stft"] / n,
                    "enh/epoch_loss_id": m["id"] / n,
                    "enh/epoch_loss_proto": m["proto"] / n,
                    "enh/epoch_loss_clean": m["clean"] / n,
                    "enh/epoch_gen_cos": m["gen_cos"] / n,
                }, commit=False)

            do_eval = ((epoch + 1) % EVAL_EVERY_N_EPOCHS == 0) or (epoch == ENH_EPOCHS - 1)
            if do_eval:
                parts = {}; wsum = 0.0
                for snr, w in VAL_SNR_WEIGHTS:
                    eer = evaluate_val(snr=snr); parts[snr] = eer; wsum += w * eer
                clean_eer = evaluate_val(snr=None)
                print(f"  Enh val weighted: {wsum:.2f}%  " +
                      "  ".join(f"{s}dB={v:.2f}" for s, v in parts.items()) + f"  clean={clean_eer:.2f}%")

                if _WANDB_ENABLED:
                    log = {"enh/epoch": epoch + 1, "enh/val_weighted": wsum, "enh/val_clean": clean_eer}
                    for s, v in parts.items(): log[f"enh/val_{s:g}dB"] = v
                    wandb.log(log, commit=True)

                if wsum < best_val:
                    best_val = wsum
                    best_state = {k: v.detach().cpu().clone() for k, v in enhancer.state_dict().items()}
                    torch.save({
                        "enhancer_state_dict": enhancer.state_dict(),
                        "best_val": best_val,
                        "config": {"LAMBDA_STFT": LAMBDA_STFT, "LAMBDA_SI": LAMBDA_SI,
                                   "LAMBDA_ID": LAMBDA_ID, "LAMBDA_PROTO": LAMBDA_PROTO,
                                   "LAMBDA_CLEAN": LAMBDA_CLEAN, "no_gate": True, "simplified_losses": True},
                    }, best_path)
                    print(f"  ✓ New best ({wsum:.2f}%)")
                    if _WANDB_ENABLED:
                        wandb.run.summary["enh_best_val"] = best_val
                        wandb.run.summary["enh_best_epoch"] = epoch + 1
                    if _DRIVE_ENABLED:
                        try: fs.put(best_path, f"{_DRIVE_REMOTE_ENH}/simple_enhancer_best.pt")
                        except Exception as e: print(f"  [drive] best upload failed: {e}")

                if USE_CUDA: torch.cuda.empty_cache()
                gc.collect()

            resume_payload = {
                "epoch": epoch,
                "enhancer_state_dict": enhancer.state_dict(),
                "optimizer_state_dict": opt.state_dict(),
                "scheduler_state_dict": sched.state_dict(),
                "scaler_state_dict": scaler.state_dict() if scaler else None,
                "best_val": best_val,
            }
            torch.save(resume_payload, local_resume)
            torch.save(resume_payload, os.path.join(ENH_CKPT_DIR, f"resume_epoch{epoch + 1}.pt"))

            if (epoch + 1) % _DRIVE_SYNC_EVERY == 0 or epoch == ENH_EPOCHS - 1:
                drive_sync(local_resume, _DRIVE_REMOTE_ENH, epoch, best_path, "simple_enhancer_best.pt")

        print(f"\nEnhancer training time: {(time.time() - t0) / 60:.1f} min")
        if best_state is not None:
            enhancer.load_state_dict(best_state)
            print(f"Loaded best enhancer (val={best_val:.2f}%)")
        elif os.path.isfile(best_path):
            ck = torch.load(best_path, map_location="cpu", weights_only=False)
            enhancer.load_state_dict(ck["enhancer_state_dict"])
            best_val = ck.get("best_val", best_val)
            print(f"Loaded best enhancer from disk (val={best_val:.2f}%)")
        enhancer.eval()
        return best_val

    # =========================================================================
    # =========== PART 2: TRAIN FUSION ON FROZEN ENHANCER ===========
    # =========================================================================
    class SelfAttentionFusion(nn.Module):
        """Blend weight beta is predicted directly from the pooled transformer
        features instead of inferred from cos(raw, enhanced). The old cosine
        proxy under-corrected specifically on babble at low SNR, because
        speech-shaped interference keeps the raw and enhanced embeddings
        looking more similar than they should even when the raw one is badly
        corrupted. beta starts at 0.5 (same as the old fixed-mean prior) and
        is free to move wherever the triplet-loss gradient sends it."""
        def __init__(self, dim=EMB_DIM, nh=SA_NUM_HEADS, nl=SA_NUM_LAYERS, dp=SA_DROPOUT):
            super().__init__()
            self.proj = nn.Linear(dim, dim)
            self.pos_emb = nn.Parameter(torch.zeros(1, 2, dim))
            layer = nn.TransformerEncoderLayer(d_model=dim, nhead=nh, dim_feedforward=dim * 4,
                                                dropout=dp, batch_first=True, activation="gelu", norm_first=True)
            self.encoder = nn.TransformerEncoder(layer, num_layers=nl)
            self.norm = nn.LayerNorm(dim)
            self.gate_head = nn.Linear(dim, 1)
            nn.init.zeros_(self.gate_head.weight); nn.init.zeros_(self.gate_head.bias)  # sigmoid(0)=0.5
            self.proj_out = nn.Linear(dim, dim)
            nn.init.zeros_(self.proj_out.weight); nn.init.zeros_(self.proj_out.bias)     # delta=0 at init
        def forward(self, en, ee, return_beta=False):
            n = self.proj(en); e = self.proj(ee)
            x = torch.stack([n, e], dim=1).contiguous() + self.pos_emb
            h = self.encoder(x)
            pooled = self.norm(h.mean(dim=1))
            beta = torch.sigmoid(self.gate_head(pooled))       # [B,1] learned blend weight
            delta = self.proj_out(pooled)                        # small residual correction
            blended = (1.0 - beta) * en + beta * ee
            out = F.normalize(blended + delta, dim=-1)
            if return_beta:
                return out, beta.squeeze(-1)
            return out

    def train_fusion():
        print("\n" + "=" * 78 + "\nPART 2: SA FUSION ON FROZEN ENHANCER\n" + "=" * 78)
        for p in enhancer.parameters(): p.requires_grad = False
        enhancer.eval()

        fusion = SelfAttentionFusion().to(DEVICE)
        print(f"Fusion params: {sum(p.numel() for p in fusion.parameters()):,}")

        train_ds = VoxDataset(train_files, spk_to_idx, train_noise, SNR_RANGE,
                               balance_noise_types=True, noise_banks=noise_train_banks)
        print("  [experiment] fusion training noise types balanced 33/33/33 "
              "(babble/music/noise), overriding the pooled-file-count default.")
        sampler = PKBatchSampler(train_ds, P_SPEAKERS, K_UTTS, FUS_STEPS_PER_EP)
        nw = 4 if _MP_CTX is not None else 0
        lk = dict(batch_sampler=sampler, num_workers=nw, pin_memory=True,
                  persistent_workers=(nw > 0), collate_fn=collate_v_simple)
        if _MP_CTX is not None: lk["multiprocessing_context"] = _MP_CTX
        loader = DataLoader(train_ds, **lk)

        opt = torch.optim.AdamW(fusion.parameters(), lr=FUS_LR, weight_decay=1e-5)
        total_steps = FUS_EPOCHS * FUS_STEPS_PER_EP
        def lr_lam(s):
            if s < FUS_WARMUP_STEPS: return s / max(1, FUS_WARMUP_STEPS)
            p = (s - FUS_WARMUP_STEPS) / max(1, total_steps - FUS_WARMUP_STEPS)
            return 0.5 * (1.0 + math.cos(math.pi * p))
        sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lam)
        scaler = make_scaler(USE_CUDA)

        local_resume = os.path.join(FUS_CKPT_DIR, "resume_latest.pt")
        best_path = os.path.join(FUS_CKPT_DIR, "simple_fusion_best.pt")
        start_epoch = 0; best_rel = float("inf")

        drive_pull_latest(local_resume, _DRIVE_REMOTE_FUS)
        if os.path.isfile(local_resume):
            try:
                ck = torch.load(local_resume, map_location="cpu", weights_only=False)
                fusion.load_state_dict(ck["fusion_state_dict"])
                opt.load_state_dict(ck["optimizer_state_dict"])
                sched.load_state_dict(ck["scheduler_state_dict"])
                if ck.get("scaler_state_dict"):
                    try: scaler.load_state_dict(ck["scaler_state_dict"])
                    except Exception: pass
                start_epoch = int(ck.get("epoch", -1)) + 1
                best_rel = float(ck.get("best_rel", float("inf")))
                print(f"[resume] fusion: epoch {start_epoch}, best_rel={best_rel:.3f}")
            except Exception as e:
                print(f"[resume] fusion: failed ({e}), starting fresh")

        def triplet_loss(anchor, fused, labels, margin=TRIPLET_MARGIN):
            B = fused.size(0); dev = fused.device
            d_pos = 1.0 - (anchor * fused).sum(-1)
            sim = anchor @ fused.t(); d_mat = 1.0 - sim
            same = labels.unsqueeze(0) == labels.unsqueeze(1); diff = ~same
            d_neg = d_mat.masked_fill(~diff, float("inf"))
            valid = (d_neg > d_pos.unsqueeze(1)) & (d_neg < (d_pos.unsqueeze(1) + margin))
            d_semi = d_neg.masked_fill(~valid, float("inf"))
            has_semi = valid.any(dim=1)
            bs, _ = d_semi.min(dim=1); hr, _ = d_neg.min(dim=1)
            d_sel = torch.where(has_semi, bs, hr)
            losses = F.relu(d_pos - d_sel + margin)
            va = torch.isfinite(d_sel)
            if va.sum() == 0: return torch.tensor(0.0, device=dev, requires_grad=True), 0
            return losses[va].mean(), int((losses[va] > 0).sum().item())

        def fus_preserve(fused, en, snrs, th=FUS_PRESERVE_SNR_THRESHOLD, tp=FUS_PRESERVE_SNR_TEMP):
            w = torch.sigmoid((snrs - th) / tp)
            per = 1.0 - (fused * en).sum(-1)
            return (per * w).sum() / w.sum().clamp_min(1e-6)

        @torch.inference_mode()
        def compute_raw_baseline():
            print("  Computing raw val baseline ...")
            enr_cache = {}
            for p in VAL_ENR:
                w = load_wav(p, SR).unsqueeze(0).to(DEVICE)
                wl = torch.ones(1, device=DEVICE)
                enr_cache[p] = ecapa_embed(w, wl).cpu()[0]
            rvb = {}
            for nt in ["babble", "music", "noise"]:
                nf = noise_train_banks[nt]
                for snr in FUS_VAL_SNRS:
                    tst_cache = {}
                    for i in range(0, len(VAL_TST), EVAL_BATCH):
                        b = VAL_TST[i:i + EVAL_BATCH]
                        wavs = [load_wav(p, SR) for p in b]
                        mixed = []
                        for p, w in zip(b, wavs):
                            r = random.Random(sseed("fusval", nt, snr, p))
                            npx = r.choice(nf); nz = load_wav(npx, SR)
                            no, _, _ = mix_det(w, nz, snr, ("fusval", nt, snr, p))
                            mixed.append(no)
                        L = torch.tensor([w.numel() for w in mixed], dtype=torch.long)
                        ml = int(L.max().item()); wl = (L.float() / ml).to(DEVICE)
                        pad = torch.stack([F.pad(w, (0, ml - w.numel())) for w in mixed]).to(DEVICE)
                        emb = ecapa_embed(pad, wl).cpu()
                        for j, p in enumerate(b): tst_cache[p] = emb[j]
                    sc = []; lb = []
                    for e, t, l in val_trials:
                        ee = enr_cache[e].to(DEVICE); te = tst_cache[t].to(DEVICE)
                        sc.append(F.cosine_similarity(ee.unsqueeze(0), te.unsqueeze(0)).item())
                        lb.append(l)
                    rvb[(nt, snr)] = compute_eer(sc, lb)
            tst_cache = {}
            for i in range(0, len(VAL_TST), EVAL_BATCH):
                b = VAL_TST[i:i + EVAL_BATCH]
                wavs = [load_wav(p, SR) for p in b]
                L = torch.tensor([w.numel() for w in wavs], dtype=torch.long)
                ml = int(L.max().item()); wl = (L.float() / ml).to(DEVICE)
                pad = torch.stack([F.pad(w, (0, ml - w.numel())) for w in wavs]).to(DEVICE)
                emb = ecapa_embed(pad, wl).cpu()
                for j, p in enumerate(b): tst_cache[p] = emb[j]
            sc = []; lb = []
            for e, t, l in val_trials:
                ee = enr_cache[e].to(DEVICE); te = tst_cache[t].to(DEVICE)
                sc.append(F.cosine_similarity(ee.unsqueeze(0), te.unsqueeze(0)).item())
                lb.append(l)
            rvc = compute_eer(sc, lb)
            print(f"  Raw baseline cached: {len(rvb)} conditions, clean {rvc:.2f}%")
            return rvb, rvc

        rvb, rvc = compute_raw_baseline()

        @torch.inference_mode()
        def validate():
            fusion.eval()
            enr_cache = {}
            for p in VAL_ENR:
                w = load_wav(p, SR).unsqueeze(0).to(DEVICE)
                wl = torch.ones(1, device=DEVICE)
                enr_cache[p] = ecapa_embed(w, wl).cpu()[0]
            per = {}
            beta_sum = {}; beta_cnt = {}
            for nt in ["babble", "music", "noise"]:
                nf = noise_train_banks[nt]
                for snr in FUS_VAL_SNRS:
                    tst_fused = {}
                    for i in range(0, len(VAL_TST), EVAL_BATCH):
                        b = VAL_TST[i:i + EVAL_BATCH]
                        wavs = [load_wav(p, SR) for p in b]
                        mixed = []
                        for p, w in zip(b, wavs):
                            r = random.Random(sseed("fusval", nt, snr, p))
                            npx = r.choice(nf); nz = load_wav(npx, SR)
                            no, _, _ = mix_det(w, nz, snr, ("fusval", nt, snr, p))
                            mixed.append(no)
                        L = torch.tensor([w.numel() for w in mixed], dtype=torch.long)
                        ml = int(L.max().item()); wl = (L.float() / ml).to(DEVICE)
                        pad = torch.stack([F.pad(w, (0, ml - w.numel())) for w in mixed]).to(DEVICE)
                        en = ecapa_embed(pad, wl)
                        ee = ecapa_embed(_enhance_chunked(pad, wl, amp=False).float(), wl)
                        f, beta = fusion(en, ee, return_beta=True)
                        f = f.cpu(); beta_sum[(nt, snr)] = beta_sum.get((nt, snr), 0.0) + beta.sum().item()
                        beta_cnt[(nt, snr)] = beta_cnt.get((nt, snr), 0) + beta.numel()
                        for j, p in enumerate(b): tst_fused[p] = f[j]
                    sc = []; lb = []
                    for e, t, l in val_trials:
                        ee = enr_cache[e].to(DEVICE); te = tst_fused[t].to(DEVICE)
                        sc.append(F.cosine_similarity(ee.unsqueeze(0), te.unsqueeze(0)).item())
                        lb.append(l)
                    fe = compute_eer(sc, lb); re = rvb[(nt, snr)]
                    per[(nt, snr)] = (fe, re, beta_sum[(nt, snr)] / max(1, beta_cnt[(nt, snr)]))
            fusion.train()
            ratios = [v[0] / max(v[1], 0.5) for v in per.values()]
            return float(np.mean(ratios)), per

        t0 = time.time()
        best_state = None
        for epoch in range(start_epoch, FUS_EPOCHS):
            fusion.train()
            m = defaultdict(float)
            pbar = tqdm(loader, desc=f"Fus {epoch + 1}/{FUS_EPOCHS}")
            opt.zero_grad(set_to_none=True)

            for step_i, (noisy, clean, labels, wl, snrs) in enumerate(pbar):
                noisy = noisy.to(DEVICE, non_blocking=True)
                clean = clean.to(DEVICE, non_blocking=True)
                labels = labels.to(DEVICE, non_blocking=True)
                wl = wl.to(DEVICE, non_blocking=True)
                snrs = snrs.to(DEVICE, non_blocking=True)

                with torch.no_grad():
                    ec = ecapa_embed(clean, wl)
                    en = ecapa_embed(noisy, wl)
                    ge = _enhance_chunked(noisy, wl, amp=False).float()
                    ee = ecapa_embed(ge, wl)

                with make_autocast(USE_CUDA):
                    f, beta = fusion(en.float(), ee.float(), return_beta=True)

                lt, na = triplet_loss(ec, f, labels)
                lp = fus_preserve(f, en, snrs)
                total = lt + LAMBDA_FUS_PRESERVE * lp
                if not torch.isfinite(total): continue

                scaler.scale(total).backward(); scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(fusion.parameters(), FUS_GRAD_CLIP)
                prev = scaler.get_scale()
                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
                if scaler.get_scale() >= prev: sched.step()

                m["total"] += total.item(); m["triplet"] += lt.item()
                m["preserve"] += lp.item(); m["active"] += na
                m["beta_mean"] += beta.mean().item()

                if _WANDB_ENABLED:
                    global_step = epoch * FUS_STEPS_PER_EP + step_i
                    wandb.log({
                        "fus/step": global_step,
                        "fus/loss_total": total.item(),
                        "fus/loss_triplet": lt.item(),
                        "fus/loss_preserve": lp.item(),
                        "fus/n_active": na,
                        "fus/beta_mean": beta.mean().item(),
                        "fus/lr": opt.param_groups[0]["lr"],
                    }, commit=True)

                pbar.set_postfix({"L": f"{total.item():.3f}", "trip": f"{lt.item():.3f}",
                                  "prsv": f"{lp.item():.3f}", "act": f"{na}", "beta": f"{beta.mean().item():.2f}"})

            n = max(1, len(pbar))
            print(f"\nFus Epoch {epoch + 1}: " + " | ".join(f"{k}={v / n:.4f}" for k, v in m.items()))

            if _WANDB_ENABLED:
                wandb.log({
                    "fus/epoch": epoch + 1,
                    "fus/epoch_loss_total": m["total"] / n,
                    "fus/epoch_loss_triplet": m["triplet"] / n,
                    "fus/epoch_loss_preserve": m["preserve"] / n,
                    "fus/epoch_n_active": m["active"] / n,
                    "fus/epoch_beta_mean": m["beta_mean"] / n,
                }, commit=False)

            do_eval = ((epoch + 1) % EVAL_EVERY_N_EPOCHS == 0) or (epoch == FUS_EPOCHS - 1)
            if do_eval:
                rv, pc = validate()
                print(f"  Mean ratio: {rv:.3f}")
                wandb_val_log = {"fus/epoch": epoch + 1, "fus/mean_ratio": rv} if _WANDB_ENABLED else None
                for nt in ["babble", "music", "noise"]:
                    cells = []
                    for snr in FUS_VAL_SNRS:
                        if (nt, snr) in pc:
                            fe, re, bm = pc[(nt, snr)]
                            cells.append(f"{snr:+5.0f}dB={fe:5.2f}|{re:5.2f}|\u03b2={bm:.2f}")
                            if _WANDB_ENABLED:
                                wandb_val_log[f"fus/eer_{nt}_{snr:g}dB"] = fe
                                wandb_val_log[f"fus/beta_{nt}_{snr:g}dB"] = bm
                    print(f"    {nt:8s}  " + "  ".join(cells))
                if _WANDB_ENABLED:
                    wandb.log(wandb_val_log, commit=True)

                if rv < best_rel:
                    best_rel = rv
                    best_state = {k: v.detach().cpu().clone() for k, v in fusion.state_dict().items()}
                    torch.save({
                        "fusion_state_dict": fusion.state_dict(),
                        "best_rel": best_rel,
                        "enhancer_source": os.path.join(ENH_CKPT_DIR, "simple_enhancer_best.pt"),
                    }, best_path)
                    print(f"  ✓ New best ({rv:.3f})")
                    if _WANDB_ENABLED:
                        wandb.run.summary["fus_best_rel"] = best_rel
                        wandb.run.summary["fus_best_epoch"] = epoch + 1
                    if _DRIVE_ENABLED:
                        try: fs.put(best_path, f"{_DRIVE_REMOTE_FUS}/simple_fusion_best.pt")
                        except Exception as e: print(f"  [drive] best upload failed: {e}")

                if USE_CUDA: torch.cuda.empty_cache()
                gc.collect()

            resume_payload = {
                "epoch": epoch,
                "fusion_state_dict": fusion.state_dict(),
                "optimizer_state_dict": opt.state_dict(),
                "scheduler_state_dict": sched.state_dict(),
                "scaler_state_dict": scaler.state_dict() if scaler else None,
                "best_rel": best_rel,
            }
            torch.save(resume_payload, local_resume)
            torch.save(resume_payload, os.path.join(FUS_CKPT_DIR, f"resume_epoch{epoch + 1}.pt"))

            if (epoch + 1) % _DRIVE_SYNC_EVERY == 0 or epoch == FUS_EPOCHS - 1:
                drive_sync(local_resume, _DRIVE_REMOTE_FUS, epoch, best_path, "simple_fusion_best.pt")

        print(f"\nFusion training time: {(time.time() - t0) / 60:.1f} min")
        if best_state is not None:
            fusion.load_state_dict(best_state)
        elif os.path.isfile(best_path):
            ck = torch.load(best_path, map_location="cpu", weights_only=False)
            fusion.load_state_dict(ck["fusion_state_dict"])
            best_rel = ck.get("best_rel", best_rel)
            print(f"Loaded best fusion from disk (rel={best_rel:.3f})")
        fusion.eval()
        return fusion

    # =========================================================================
    # WANDB INIT (single run spans both stages; enh/*, fus/*, test/* prefixes)
    # =========================================================================
    if _WANDB_ENABLED:
        wandb.init(
            project=os.environ.get("WANDB_PROJECT", "waveunet-speaker-ft"),
            name=os.environ.get("WANDB_RUN_NAME", f"v6-final-{int(time.time())}"),
            config={
                "enh_epochs": ENH_EPOCHS, "enh_lr": ENH_LR,
                "lambda_stft": LAMBDA_STFT, "lambda_si": LAMBDA_SI, "lambda_id": LAMBDA_ID,
                "lambda_proto": LAMBDA_PROTO, "lambda_clean": LAMBDA_CLEAN,
                "fus_epochs": FUS_EPOCHS, "fus_lr": FUS_LR,
                "triplet_margin": TRIPLET_MARGIN, "lambda_fus_preserve": LAMBDA_FUS_PRESERVE,
                "fusion_arch": "learned_blend_beta",  # vs. v5's cosine-gated alpha
                "fusion_noise_sampling": "balanced_33_33_33",  # vs. v5's pooled-file-count
                "no_gate": True, "no_arcface": True, "no_dfl": True, "no_gen_imp_margin": True,
                "device": DEVICE,
            },
            tags=["v6", "final", "simplified", "balanced-noise", "learned-blend"],
        )
        wandb.define_metric("enh/step"); wandb.define_metric("enh/*", step_metric="enh/step")
        wandb.define_metric("fus/step"); wandb.define_metric("fus/*", step_metric="fus/step")

    # =========================================================================
    # RUN TRAINING
    # =========================================================================
    enh_val = train_enhancer()
    fusion = train_fusion()

    # =========================================================================
    # FULL TEST SWEEP (raw vs. enhancer-only vs. fused, matching v4_long's format
    # plus a fused_eer column for the actually-deployed pipeline)
    # =========================================================================
    print("\n" + "=" * 78 + "\nFULL TEST\n" + "=" * 78)
    if not os.path.isfile(TRIAL_FILE):
        urllib.request.urlretrieve(
            "https://www.robots.ox.ac.uk/~vgg/data/voxceleb/meta/veri_test2.txt", TRIAL_FILE)

    def load_trials(path):
        out = []
        with open(path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 3: out.append((parts[1], parts[2], int(parts[0])))
        return out
    def resolve(rel):
        rel = rel.replace("\\", "/").lstrip("/")
        for base in (TEST_ROOT, os.path.join(TEST_ROOT, "wav"), "/marimo/voxceleb1_test"):
            p = os.path.join(base, rel)
            if os.path.isfile(p): return p
        mm = list(Path(TEST_ROOT).rglob(os.path.basename(rel)))
        return str(mm[0]) if mm else None

    all_trials = load_trials(TRIAL_FILE)
    resolved = []
    for e, t, lab in all_trials:
        ep, tp = resolve(e), resolve(t)
        if ep and tp: resolved.append((ep, tp, lab))
    assert not [(e, t) for e, t, _ in resolved if e == t]
    ue = sorted({e for e, _, _ in resolved}); ut = sorted({t for _, t, _ in resolved})
    print(f"Resolved: {len(resolved)}  Enroll: {len(ue)}  Test: {len(ut)}")

    @torch.inference_mode()
    def embed_clean(paths, batch_size=16):
        cache = {}
        for i in range(0, len(paths), batch_size):
            batch = paths[i:i + batch_size]
            wavs = [load_wav(p, SR) for p in batch]
            L = torch.tensor([w.numel() for w in wavs], dtype=torch.long)
            ml = int(L.max().item()); wl = (L.float() / ml).to(DEVICE)
            pad = torch.stack([F.pad(w, (0, ml - w.numel())) for w in wavs]).to(DEVICE)
            emb = ecapa_embed(pad, wl).cpu()
            for j, p in enumerate(batch): cache[p] = emb[j]
        return cache

    @torch.inference_mode()
    def embed_paths_full(paths, nt=None, snr=None, noise_files=None, batch_size=16, tag="test"):
        """One deterministic noisy mixture per utterance -> raw / enh / fused embeddings."""
        raw_cache, enh_cache, fused_cache = {}, {}, {}
        for i in range(0, len(paths), batch_size):
            batch = paths[i:i + batch_size]
            wavs = [load_wav(p, SR) for p in batch]
            if snr is not None:
                mixed = []
                for p, w in zip(batch, wavs):
                    r = random.Random(sseed(tag, nt, snr, p))
                    npx = r.choice(noise_files); nz = load_wav(npx, SR)
                    no, _, _ = mix_det(w, nz, snr, (tag, nt, snr, p))
                    mixed.append(no)
                wavs = mixed
            L = torch.tensor([w.numel() for w in wavs], dtype=torch.long)
            ml = int(L.max().item()); wl = (L.float() / ml).to(DEVICE)
            pad = torch.stack([F.pad(w, (0, ml - w.numel())) for w in wavs]).to(DEVICE)

            en = ecapa_embed(pad, wl)
            enh_wav = _enhance_chunked(pad, wl, amp=False).float()
            ee = ecapa_embed(enh_wav, wl)
            fe = fusion(en, ee)

            raw_emb = en.cpu(); enh_emb = ee.cpu(); fused_emb = fe.cpu()
            for j, p in enumerate(batch):
                raw_cache[p] = raw_emb[j]; enh_cache[p] = enh_emb[j]; fused_cache[p] = fused_emb[j]
            del pad, enh_wav
        if USE_CUDA: torch.cuda.empty_cache()
        return raw_cache, enh_cache, fused_cache

    print("\nCaching clean enrollment embeddings ...")
    enr_cache = embed_clean(ue, batch_size=16)
    print(f"  {len(enr_cache)} cached")

    conditions = [("clean", None, None)]
    for nt, _ in NOISE_SUBDIRS:
        for snr in EVAL_SNRS: conditions.append((nt, nt, snr))
    print(f"\nEvaluating {len(conditions)} conditions ...")
    results = []
    t0 = time.time()
    for cond, nt, snr in tqdm(conditions, desc="conditions"):
        nf = noise_test_banks[nt] if snr is not None else None
        raw_cache, enh_cache, fused_cache = embed_paths_full(ut, nt=nt, snr=snr, noise_files=nf, batch_size=16)

        s_raw, s_enh, s_fused, lb = [], [], [], []
        for e, t, l in resolved:
            ee = enr_cache[e].to(DEVICE)
            s_raw.append(F.cosine_similarity(ee.unsqueeze(0), raw_cache[t].unsqueeze(0).to(DEVICE)).item())
            s_enh.append(F.cosine_similarity(ee.unsqueeze(0), enh_cache[t].unsqueeze(0).to(DEVICE)).item())
            s_fused.append(F.cosine_similarity(ee.unsqueeze(0), fused_cache[t].unsqueeze(0).to(DEVICE)).item())
            lb.append(l)
        s_raw = np.array(s_raw); s_enh = np.array(s_enh); s_fused = np.array(s_fused); lb = np.array(lb)
        raw_eer = compute_eer(s_raw, lb); enh_eer = compute_eer(s_enh, lb); fused_eer = compute_eer(s_fused, lb)
        results.append({
            "condition": cond, "noise_type": nt if snr is not None else "clean", "snr": snr,
            "raw_eer": raw_eer, "enh_eer": enh_eer, "fused_eer": fused_eer,
            "delta_enh_eer": enh_eer - raw_eer, "delta_fused_eer": fused_eer - raw_eer,
        })
        gc.collect()
        if USE_CUDA: torch.cuda.empty_cache()

    print(f"Sweep time: {(time.time() - t0) / 60:.1f} min")
    df = pd.DataFrame(results)
    out_csv = "/marimo/eval_waveunet_ft_v6_final.csv"
    df.to_csv(out_csv, index=False)
    if _DRIVE_ENABLED:
        try:
            fs.put(out_csv, f"{_DRIVE_REMOTE_FUS}/eval_waveunet_ft_v6_final.csv")
            print("[drive] uploaded eval CSV")
        except Exception as e:
            print(f"[drive] eval CSV upload failed: {e}")

    def piv(c):
        p = df.pivot_table(index="noise_type", columns="snr", values=c, dropna=False)
        return p.reindex(["clean"] + [c for c in p.index if c != "clean"])
    print("\n" + "=" * 78 + "\nEER Raw ECAPA\n" + "=" * 78); print(piv("raw_eer").round(2))
    print("\n" + "=" * 78 + "\nEER Enhancer-only (v5 simplified)\n" + "=" * 78); print(piv("enh_eer").round(2))
    print("\n" + "=" * 78 + "\nEER Fused (deployed pipeline)\n" + "=" * 78); print(piv("fused_eer").round(2))
    print("\n" + "=" * 78 + "\nΔ EER (Enhanced − Raw)\n" + "=" * 78); print(piv("delta_enh_eer").round(3))
    print("\n" + "=" * 78 + "\nΔ EER (Fused − Raw)\n" + "=" * 78); print(piv("delta_fused_eer").round(3))

    clean_row = df[df["condition"] == "clean"].iloc[0]
    print(f"\nClean: raw={clean_row['raw_eer']:.2f}%  enh={clean_row['enh_eer']:.2f}%  "
          f"fused={clean_row['fused_eer']:.2f}%  Δenh={clean_row['delta_enh_eer']:+.2f}%  "
          f"Δfused={clean_row['delta_fused_eer']:+.2f}%")
    nd = df[df["condition"] != "clean"]
    for nt in nd["noise_type"].unique():
        s = nd[nd["noise_type"] == nt]
        print(f"  {nt:8s}  raw={s['raw_eer'].mean():5.2f}%  enh={s['enh_eer'].mean():5.2f}%  "
              f"fused={s['fused_eer'].mean():5.2f}%  Δenh={s['delta_enh_eer'].mean():+.2f}%  "
              f"Δfused={s['delta_fused_eer'].mean():+.2f}%")
    print(f"  {'ALL':8s}  raw={nd['raw_eer'].mean():5.2f}%  enh={nd['enh_eer'].mean():5.2f}%  "
          f"fused={nd['fused_eer'].mean():5.2f}%  Δenh={nd['delta_enh_eer'].mean():+.2f}%  "
          f"Δfused={nd['delta_fused_eer'].mean():+.2f}%")
    pp = nd[nd["snr"].isin(PAPER_SNRS)]
    print(f"\nPaper subset: raw={pp['raw_eer'].mean():.2f}%  enh={pp['enh_eer'].mean():.2f}%  "
          f"fused={pp['fused_eer'].mean():.2f}%")

    if _WANDB_ENABLED:
        wandb.run.summary["test_clean_raw_eer"] = float(clean_row["raw_eer"])
        wandb.run.summary["test_clean_fused_eer"] = float(clean_row["fused_eer"])
        wandb.run.summary["test_all_raw_eer"] = float(nd["raw_eer"].mean())
        wandb.run.summary["test_all_enh_eer"] = float(nd["enh_eer"].mean())
        wandb.run.summary["test_all_fused_eer"] = float(nd["fused_eer"].mean())
        wandb.run.summary["test_paper_raw_eer"] = float(pp["raw_eer"].mean())
        wandb.run.summary["test_paper_fused_eer"] = float(pp["fused_eer"].mean())
        for nt in nd["noise_type"].unique():
            s = nd[nd["noise_type"] == nt]
            wandb.run.summary[f"test_{nt}_delta_fused_eer"] = float(s["delta_fused_eer"].mean())
        test_table = wandb.Table(
            columns=["noise_type", "snr", "raw_eer", "enh_eer", "fused_eer", "delta_fused_eer"],
            data=df[["noise_type", "snr", "raw_eer", "enh_eer", "fused_eer", "delta_fused_eer"]].values.tolist(),
        )
        wandb.log({"test/per_condition": test_table})
        wandb.finish()

    return df, enh_val


finetune_waveunet_v6_final()

# Speaker Verification — Training & Evaluation Log

**Device:** NVIDIA RTX PRO 6000 Blackwell Server Edition (102.0 GB VRAM) | CUDA

## Data Setup

**MUSAN split**

| Noise type | Train | Test |
|------------|------:|-----:|
| babble     |   299 |  127 |
| music      |   462 |  198 |
| noise      |   651 |  279 |

**VoxCeleb1 dev**
- Train: 1171 speakers
- Val: 40 speakers
- Utterances: 143,328
- Val trials: 320

**Models**
- ECAPA: embedding dim = 192
- Wave-U-Net: `missing=2, unexpected=2`

## Training

### Simplified Enhancer (5 losses, no gate)

Loss weights:
- STFT = 0.4
- SI = 0.05
- ID = 0.4
- PROTO = 0.3
- CLEAN = 0.15

Resumed from `simple_enhancer_v5/resume_epoch30.pt` (epoch 30, best_val = 5.50).

| Stage | Time |
|-------|-----:|
| Enhancer training | 130.2 min |
| Fusion training | 115.6 min |
| Sweep | 40.6 min |

Loaded best enhancer (val = 5.41%).

## EER Results

### Raw ECAPA

| noise_type | -20.0 | -15.0 | -10.0 | -5.0 | 0.0 | 5.0 | 10.0 | 15.0 | 20.0 | clean |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| babble | 46.23 | 42.49 | 35.07 | 23.07 | 10.55 | 4.29 | 1.86 | 1.16 | 0.98 | — |
| music | 39.90 | 32.94 | 22.37 | 9.90 | 3.67 | 1.71 | 1.28 | 1.05 | 0.94 | — |
| noise | 24.36 | 16.24 | 8.71 | 3.98 | 2.17 | 1.41 | 1.16 | 1.07 | 0.99 | — |
| clean | — | — | — | — | — | — | — | — | — | 0.90 |

### Enhancer-only (v5 simplified)

| noise_type | -20.0 | -15.0 | -10.0 | -5.0 | 0.0 | 5.0 | 10.0 | 15.0 | 20.0 | clean |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| babble | 34.26 | 25.63 | 18.11 | 12.75 | 7.86 | 4.69 | 2.43 | 1.48 | 1.03 | — |
| music | 38.58 | 31.05 | 19.47 | 8.21 | 3.28 | 1.72 | 1.28 | 1.03 | 0.97 | — |
| noise | 23.88 | 16.27 | 9.22 | 4.77 | 2.51 | 1.58 | 1.23 | 1.12 | 0.99 | — |
| clean | — | — | — | — | — | — | — | — | — | 0.91 |

### Fused (deployed pipeline)

| noise_type | -20.0 | -15.0 | -10.0 | -5.0 | 0.0 | 5.0 | 10.0 | 15.0 | 20.0 | clean |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| babble | 32.56 | 23.38 | 16.33 | 11.30 | 7.06 | 3.99 | 2.01 | 1.29 | 1.02 | — |
| music | 34.72 | 26.03 | 16.14 | 6.87 | 3.04 | 1.72 | 1.28 | 1.04 | 0.96 | — |
| noise | 18.00 | 11.58 | 6.71 | 3.52 | 2.07 | 1.47 | 1.18 | 1.12 | 1.03 | — |
| clean | — | — | — | — | — | — | — | — | — | 0.94 |

### Δ EER (Enhanced − Raw)

| noise_type | -20.0 | -15.0 | -10.0 | -5.0 | 0.0 | 5.0 | 10.0 | 15.0 | 20.0 | clean |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| babble | -11.970 | -16.859 | -16.952 | -10.313 | -2.696 | 0.396 | 0.577 | 0.319 | 0.045 | — |
| music | -1.324 | -1.893 | -2.898 | -1.691 | -0.388 | 0.005 | -0.005 | -0.019 | 0.037 | — |
| noise | -0.484 | 0.032 | 0.505 | 0.787 | 0.335 | 0.170 | 0.077 | 0.051 | -0.003 | — |
| clean | — | — | — | — | — | — | — | — | — | 0.005 |

### Δ EER (Fused − Raw)

| noise_type | -20.0 | -15.0 | -10.0 | -5.0 | 0.0 | 5.0 | 10.0 | 15.0 | 20.0 | clean |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| babble | -13.669 | -19.111 | -18.734 | -11.765 | -3.496 | -0.303 | 0.157 | 0.138 | 0.035 | — |
| music | -5.179 | -6.913 | -6.227 | -3.026 | -0.630 | 0.005 | -0.003 | -0.003 | 0.027 | — |
| noise | -6.360 | -4.653 | -1.999 | -0.463 | -0.104 | 0.061 | 0.024 | 0.053 | 0.043 | — |
| clean | — | — | — | — | — | — | — | — | — | 0.035 |

## Summary

| Condition | Raw | Enhancer | Fused | Δ Enh | Δ Fused |
|---|---:|---:|---:|---:|---:|
| Clean | 0.90% | 0.91% | 0.94% | +0.01% | +0.03% |
| Babble | 18.41% | 12.03% | 10.99% | −6.38% | −7.42% |
| Music | 12.64% | 11.73% | 10.20% | −0.91% | −2.44% |
| Noise | 6.68% | 6.84% | 5.19% | +0.16% | −1.49% |
| **ALL** | **12.58%** | **10.20%** | **8.79%** | **−2.38%** | **−3.78%** |

**Paper subset:** raw = 21.44% | enh = 17.06% | fused = 14.62%

# Final Results — Simplified WaveUNet Enhancer + Self-Attention Fusion (v6)

## Pipeline summary

- **Enhancer**: WaveUNet fine-tuned with 5 losses (STFT, SI-SDR, ID, PROTO, CLEAN). No gate, no ArcFace, no DFL, no GEN/IMP/MARGIN losses.
- **Fusion**: small self-attention module that directly predicts a per-example blend weight β ∈ [0,1] between the raw and enhanced ECAPA embeddings (`beta = sigmoid(gate_head(pooled))`), instead of inferring it from cosine agreement.
- **Training data**: fusion-stage noise sampled uniformly across babble/music/noise (33/33/33), correcting an imbalance in the pooled MUSAN file counts.
- **Evaluated on**: VoxCeleb1-O (`veri_test`), 37,611 resolved trials, clean + 3 noise types × 9 SNRs (−20 to +20 dB).

Enhancer checkpoint: `simple_enhancer_v5/simple_enhancer_best.pt` (val EER 5.41%)
Fusion checkpoint: `sa_fusion_v6_final/simple_fusion_best.pt` (mean ratio 0.516)

## EER by condition (%)

### Raw ECAPA (no processing)

| Noise type | −20dB | −15dB | −10dB | −5dB | 0dB | +5dB | +10dB | +15dB | +20dB |
|---|---|---|---|---|---|---|---|---|---|
| babble | 46.23 | 42.49 | 35.07 | 23.07 | 10.55 | 4.29 | 1.86 | 1.16 | 0.98 |
| music | 39.90 | 32.94 | 22.37 | 9.90 | 3.67 | 1.71 | 1.28 | 1.05 | 0.94 |
| noise | 24.36 | 16.24 | 8.71 | 3.98 | 2.17 | 1.41 | 1.16 | 1.07 | 0.99 |

Clean: **0.90%**

### Enhancer-only (pre-fusion)

| Noise type | −20dB | −15dB | −10dB | −5dB | 0dB | +5dB | +10dB | +15dB | +20dB |
|---|---|---|---|---|---|---|---|---|---|
| babble | 34.26 | 25.63 | 18.11 | 12.75 | 7.86 | 4.69 | 2.43 | 1.48 | 1.03 |
| music | 38.58 | 31.05 | 19.47 | 8.21 | 3.28 | 1.72 | 1.28 | 1.03 | 0.97 |
| noise | 23.88 | 16.27 | 9.22 | 4.77 | 2.51 | 1.58 | 1.23 | 1.12 | 0.99 |

Clean: **0.91%**

### Fused (deployed pipeline)

| Noise type | −20dB | −15dB | −10dB | −5dB | 0dB | +5dB | +10dB | +15dB | +20dB |
|---|---|---|---|---|---|---|---|---|---|
| babble | 32.56 | 23.38 | 16.33 | 11.30 | 7.06 | 3.99 | 2.01 | 1.29 | 1.02 |
| music | 34.72 | 26.03 | 16.14 | 6.87 | 3.04 | 1.72 | 1.28 | 1.04 | 0.96 |
| noise | 18.00 | 11.58 | 6.71 | 3.52 | 2.07 | 1.47 | 1.18 | 1.12 | 1.03 |

Clean: **0.94%**

### Δ EER — Fused vs. Raw

| Noise type | −20dB | −15dB | −10dB | −5dB | 0dB | +5dB | +10dB | +15dB | +20dB |
|---|---|---|---|---|---|---|---|---|---|
| babble | −13.67 | −19.11 | −18.73 | −11.77 | −3.50 | −0.30 | +0.16 | +0.14 | +0.04 |
| music | −5.18 | −6.91 | −6.23 | −3.03 | −0.63 | +0.01 | 0.00 | 0.00 | +0.03 |
| noise | −6.36 | −4.65 | −2.00 | −0.46 | −0.10 | +0.06 | +0.02 | +0.05 | +0.04 |

Negative = fusion improves over raw. Fused beats (or matches) enhancer-only at **every** SNR across all three noise types — no crossover regression anywhere in the grid.

## Summary

| | Raw | Enhancer-only | Fused | Δ (fused − raw) |
|---|---|---|---|---|
| Clean | 0.90% | 0.91% | 0.94% | +0.03% |
| babble (avg) | 18.41% | 12.03% | 10.99% | −7.42% |
| music (avg) | 12.64% | 11.73% | 10.20% | −2.44% |
| noise (avg) | 6.68% | 6.84% | 5.19% | −1.49% |
| **ALL noisy** | **12.58%** | **10.20%** | **8.79%** | **−3.78%** |
| **Paper subset** (0,−5,−10,−15,−20 dB) | **21.44%** | **17.06%** | **14.62%** | — |
